**Colab-ready:** mount Drive on demand and locate `MyDrive/CMPS355_Research` automatically. This notebook does not train models.

# 00 — Setup and project check

**Scientific purpose.** Ensure that we are using the fresh, independent workspace, the correct accepted input dataset, the expected Python packages, and writable output folders **before any analysis**.

**Input:** the `CMPS355_Research/` project directory. **Output:** a timestamped setup report under `runs/setup/`.

This notebook performs no RNA-seq preprocessing, feature selection, model training, or literature reproduction.

## A. Locate the project and import the *new* validation module

The root marker prevents accidentally running against a historical workspace. The code below only imports `src/cmps355/project_checks.py`, which was independently written for this project.

In [ ]:
from pathlib import Path
import os
import sys

# Colab runs notebooks with /content as its working directory, not the notebook's Drive folder.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')  # Explicit Google authorization, never unattended

# An environment override supports a clone, a shared drive, or another folder name.
candidates = []
if os.environ.get('CMPS355_ROOT'):
    candidates.append(Path(os.environ['CMPS355_ROOT']).expanduser())
if IN_COLAB:
    candidates += [Path('/content/drive/MyDrive/CMPS355_Research'),
                   Path('/content/CMPS355_Research')]
candidates += [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'CMPS355_Research']
ROOT = next((p.resolve() for p in candidates if (p / '.cmps355_root').is_file()), None)
if ROOT is None:
    raise FileNotFoundError(
        'Project root not found. On Colab, upload/extract CMPS355_Research under '
        'MyDrive, or set os.environ["CMPS355_ROOT"] to the actual project folder. '
        'On local Jupyter, launch it from the project root.'
    )
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('Runtime:', 'Google Colab' if IN_COLAB else 'Local Jupyter/Python')
print('Project:', ROOT)

from cmps355.project_checks import collect_report, write_report


### Install/verify scientific dependencies

Only NumPy and pandas are required for Stage 1. Colab has its own Jupyter frontend; local JupyterLab and R are **not** required here. Pinned versions support consistency across runs. If Colab requests a runtime restart after changing installed packages, restart and rerun from the beginning.

In [ ]:
# Install only when the *installed* scientific package versions differ from the tested ones.
# This also works on an ordinary local Jupyter runtime (network access if installation is needed).
from importlib.metadata import version, PackageNotFoundError
import subprocess

expected = {'numpy': '2.3.5', 'pandas': '2.2.3'}
found = {}
for package in expected:
    try:
        found[package] = version(package)
    except PackageNotFoundError:
        found[package] = None
needs_install = any(found[p] != expected[p] for p in expected)
if needs_install:
    print('Version mismatch:', found, '->', expected)
    print('Installing stage-one dependencies. If imports were already loaded, restart the runtime afterward.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt')], check=True)
else:
    print('Scientific dependencies already match tested versions:', found)


## B. Test the scientific input and software environment

Missing Python dependencies or a missing/mismatched dataset are **failures**. R is only required for authentic StableMate/TENTACLES work beginning in Stage 2, so a missing R installation is currently a **warning**.

In [ ]:
import pandas as pd

report = collect_report(ROOT)
check_table = pd.DataFrame(report["checks"])
display(check_table)
print("\nSummary:", report["summary"])
print("System:", report["platform"])
print("This notebook does not claim any model was trained.")

## C. Save the report and stop on failures

Every run creates a separate dated JSON report; no earlier report is overwritten. If the status is FAIL, inspect the table above. Common repairs: reinstall `requirements.txt`, restore the accepted dataset ZIP, or extract the project again without renaming folders.

In [ ]:
report_path = write_report(report, ROOT)
print("Saved report:", report_path.relative_to(ROOT))
if report["summary"]["FAIL"]:
    raise RuntimeError(
        f"STAGE 1 SETUP: FAIL — {report['summary']['FAIL']} failed checks. "
        "Fix the checks marked FAIL and rerun this notebook."
    )
print("STAGE 1 SETUP: PASS")
if report["summary"]["WARN"]:
    print("Warnings (not blocking Stage 1):", report["summary"]["WARN"])
print("Next: run 01_data_validation.ipynb.")